# ForkWise MLOps — Reserve & Provision

End-to-end provisioner. Run all cells top to bottom for a fresh lease.

**What this notebook does:**
1. Reserve a 24-hour KVM@TACC lease
2. Install Terraform, provision 3 VMs + network + floating IP
3. Print commands to run from your Windows machine and on node1

**Prerequisites (already in /work):**
- `clouds.yaml` — KVM@TACC application credentials
- SSH key `forkwise-key` registered on Chameleon
- On your Windows machine: `C:\Users\Krishan Guta\.ssh\forkwise_key`

**Teardown is at the bottom — uncomment and run.**

## Step 1: Configure Chameleon Context

In [1]:
import sys
print(sys.executable)

C:\Program Files\Python312\python.exe


In [2]:
import sys

packages = [
    "python-chi",
    "python-blazarclient",
    "ipywidgets",
    "ipydatagrid",
    "pandas",
    "python-openstackclient",
    "python-neutronclient",
    "python-novaclient",
    "python-glanceclient",
]

for pkg in packages:
    !"{sys.executable}" -m pip install --upgrade {pkg}

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 23.3.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/50.7 kB ? eta -:--:--
   -------- ------------------------------- 10.2/50.7 kB ? eta -:--:--
   -------------------------------- ------- 41.0/50.7 kB 393.8 kB/s eta 0:00:01
   ---------------------------------------- 50.7/50.7 kB 323.9 kB/s eta 0:00:00



[notice] A new release of pip is available: 23.3.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Defaulting to user installation because normal site-packages is not writeable


...


[notice] A new release of pip is available: 23.3.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 23.3.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 23.3.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
from chi import server, context, lease, network
import chi, os, time, datetime, subprocess, shutil, json

context.version = "1.0"
context.choose_project()
context.choose_site(default="KVM@TACC")

# ---- Project configuration ----
PROJECT_PREFIX = "proj01"
SSH_KEY_NAME   = "forkwise-key"
LEASE_END      = datetime.datetime.now(datetime.timezone.utc) + datetime.timedelta(hours=24)
VM_FLAVOR      = "m1.xlarge"
VM_COUNT       = 3
VM_IMAGE       = "CC-Ubuntu24.04"

print(f"Lease will end: {LEASE_END.isoformat()}")

MissingRequiredOptions: Auth plugin requires parameters which were not given: auth_url

## Step 2: Install Terraform

In [ ]:
TF_VERSION = "1.14.4"

commands = [
    "mkdir -p /work/.local/bin",
    f"wget -q https://releases.hashicorp.com/terraform/{TF_VERSION}/terraform_{TF_VERSION}_linux_amd64.zip",
    f"unzip -o -q terraform_{TF_VERSION}_linux_amd64.zip",
    "mv terraform /work/.local/bin",
    f"rm terraform_{TF_VERSION}_linux_amd64.zip",
]

for cmd in commands:
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if result.returncode != 0:
        print(f"Error: {cmd}\n{result.stderr}")
    else:
        print(f"Done: {cmd}")

os.environ["PATH"] = "/work/.local/bin:" + os.environ["PATH"]

result = subprocess.run("terraform --version", shell=True, capture_output=True, text=True)
print(result.stdout.split('\n')[0])

## Step 3: Create Lease

In [ ]:
LEASE_NAME = f"lease-infra-{PROJECT_PREFIX}"

l = lease.Lease(
    LEASE_NAME,
    end_date=LEASE_END,
)
l.add_flavor_reservation(
    id=chi.server.get_flavor_id(VM_FLAVOR),
    amount=VM_COUNT,
)
l.submit(idempotent=True)

reservation_id = l.get_reserved_flavors()[0].id
print(f"Lease: {LEASE_NAME} — Status: ACTIVE")
print(f"Reservation flavor ID: {reservation_id}")

## Step 4: Set Up Terraform Files

Clones the infra repo and copies `clouds.yaml` into the Terraform directory.

**Resources created by Terraform:**
- Private network + subnet (192.168.1.0/24, no gateway)
- 3 ports on private network (fixed IPs, no port security)
- 3 ports on sharednet1 (with security groups)
- 3 compute instances (CC-Ubuntu24.04, m1.xlarge)
- 1 floating IP assigned to node1

In [ ]:
INFRA_REPO = "https://github.com/Krishan101/mlops-forkwise.git"
infra_dir = "/work/mlops-forkwise"
tf_dir = f"{infra_dir}/tf/kvm"

# Clone or pull the repo
if os.path.exists(infra_dir):
    result = subprocess.run("git pull", shell=True, cwd=infra_dir, capture_output=True, text=True)
    print(f"Repo updated: {result.stdout.strip()}")
else:
    result = subprocess.run(f"git clone {INFRA_REPO} {infra_dir}", shell=True, capture_output=True, text=True)
    print(f"Repo cloned: {result.stdout.strip()}")

# Copy clouds.yaml into tf directory (secret — not in repo)
shutil.copy("/work/clouds.yaml", f"{tf_dir}/clouds.yaml")
print(f"clouds.yaml copied to {tf_dir}")

# Verify Terraform files exist
tf_files = [f for f in os.listdir(tf_dir) if f.endswith('.tf')]
print(f"Terraform files found: {', '.join(sorted(tf_files))}")

## Step 5: Terraform Init, Plan, and Apply

In [ ]:
# Build a clean environment — remove Chameleon Jupyter's OS_ vars
# which conflict with our clouds.yaml (they point to CHI@UC)
clean_env = {k: v for k, v in os.environ.items() if not k.startswith("OS_")}
clean_env["OS_CLOUD"]        = "openstack"
clean_env["PATH"]            = "/work/.local/bin:" + clean_env.get("PATH", "")
clean_env["HOME"]            = os.environ.get("HOME", "/home/jovyan")
clean_env["TF_VAR_suffix"]   = PROJECT_PREFIX
clean_env["TF_VAR_key"]      = SSH_KEY_NAME
clean_env["TF_VAR_reservation"] = reservation_id

def run_tf(command, description):
    print(f"\n{'='*60}")
    print(f"  {description}")
    print(f"{'='*60}")
    result = subprocess.run(
        command, shell=True, cwd=tf_dir, env=clean_env,
        capture_output=True, text=True
    )
    print(result.stdout)
    if result.stderr:
        print(result.stderr)
    if result.returncode != 0:
        raise Exception(f"{description} failed with return code {result.returncode}")
    return result.returncode

def _os(cmd):
    r = subprocess.run(cmd, shell=True, env=clean_env,
                       capture_output=True, text=True)
    return r.returncode, r.stdout.strip(), r.stderr.strip()

run_tf("terraform init", "Terraform Init")
run_tf("terraform validate", "Terraform Validate")
run_tf("terraform plan", "Terraform Plan")

In [ ]:
# Ensure required security groups exist
REQUIRED_SGS = {
    "allow-ssh": 22,
    "allow-30900": 30900,
    "allow-30808": 30808,
    "allow-30500": 30500,
    "allow-30300": 30300,
}

rc, existing, _ = _os("openstack security group list -f value -c Name")
existing = set(existing.splitlines())
for name, port in REQUIRED_SGS.items():
    if name in existing:
        print(f"  [ok]      {name}")
        continue
    print(f"  [create]  {name}  (tcp/{port})")
    _os(f"openstack security group create {name} --description 'auto: forkwise bringup'")
    _os(f"openstack security group rule create --protocol tcp "
        f"--dst-port {port} --remote-ip 0.0.0.0/0 {name}")
print("\nAll required security groups present.")

In [ ]:
# Clean up stale DOWN ports from previous leases if any
_os("openstack port list --status DOWN -f value -c ID | xargs -r -n1 openstack port delete")

# Apply — creates all resources
run_tf("terraform apply -auto-approve", "Terraform Apply")

# Extract outputs
result = subprocess.run(
    "terraform output -json",
    shell=True, cwd=tf_dir, env=clean_env,
    capture_output=True, text=True
)
outputs = json.loads(result.stdout)
floating_ip = outputs["floating_ip"]["value"]

print(f"\n{'='*72}")
print(f"  Infrastructure provisioned successfully!")
print(f"{'='*72}")
print(f"\n  Floating IP: {floating_ip}")

## Step 6: Next Steps

Run these commands manually after Terraform completes.

All commands are single lines — paste directly into PowerShell or the node1 terminal.

In [ ]:
print(f"{'='*72}")
print(f"  NEXT STEPS")
print(f"{'='*72}")
print()

print("STEP A — From PowerShell on your Windows machine, copy SSH key to node1:")
print()
print(f'   scp -i "C:\\Users\\Krishan Guta\\.ssh\\forkwise_key" "C:\\Users\\Krishan Guta\\.ssh\\forkwise_key" cc@{floating_ip}:~/.ssh/forkwise-key')
print()

print("STEP B — From PowerShell, copy clouds.yaml to node1:")
print()
print(f'   ssh -i "C:\\Users\\Krishan Guta\\.ssh\\forkwise_key" cc@{floating_ip} "mkdir -p ~/.config/openstack"')
print(f'   scp -i "C:\\Users\\Krishan Guta\\.ssh\\forkwise_key" "C:\\Users\\Krishan Guta\\clouds.yaml" cc@{floating_ip}:~/.config/openstack/clouds.yaml')
print()
print("   (adjust the source path if your clouds.yaml is elsewhere)")
print()

print(f"STEP C — From PowerShell, SSH into node1:")
print()
print(f'   ssh -i "C:\\Users\\Krishan Guta\\.ssh\\forkwise_key" cc@{floating_ip}')
print()

print("STEP D — On node1, set up SSH keys:")
print()
print("   chmod 600 ~/.ssh/*")
print('   ssh-keygen -t rsa -b 4096 -f ~/.ssh/id_rsa -q -N ""')
print("   cat ~/.ssh/id_rsa.pub >> ~/.ssh/authorized_keys")
print('   cat ~/.ssh/id_rsa.pub | ssh -i ~/.ssh/forkwise-key -o StrictHostKeyChecking=no cc@192.168.1.12 "cat >> ~/.ssh/authorized_keys"')
print('   cat ~/.ssh/id_rsa.pub | ssh -i ~/.ssh/forkwise-key -o StrictHostKeyChecking=no cc@192.168.1.13 "cat >> ~/.ssh/authorized_keys"')
print()

print("STEP E — On node1, disable IPv6 on all nodes:")
print()
print("   for ip in 192.168.1.11 192.168.1.12 192.168.1.13; do ssh -o StrictHostKeyChecking=no cc@$ip 'sudo sysctl -w net.ipv6.conf.ens3.disable_ipv6=1'; done")
print()

print("STEP F — On node1, install kubespray:")
print()
print("   sudo apt update && sudo apt install -y python3-pip python3-venv git tmux")
print("   git clone -b release-2.26 https://github.com/kubernetes-sigs/kubespray.git")
print("   cd kubespray")
print("   python3 -m venv .venv && source .venv/bin/activate")
print("   pip install -r requirements.txt ruamel.yaml")
print()

print("STEP G — On node1, build kubespray inventory:")
print()
print("   cp -rfp inventory/sample inventory/mycluster")
print("   CONFIG_FILE=inventory/mycluster/hosts.yaml python3 contrib/inventory_builder/inventory.py 192.168.1.11 192.168.1.12 192.168.1.13")
print("   python3 -c \"import yaml; p='inventory/mycluster/hosts.yaml'; d=yaml.safe_load(open(p)); d['all']['children']['kube_control_plane']['hosts']={'node1': None}; open(p,'w').write(yaml.safe_dump(d, default_flow_style=False))\"")
print()

print("STEP H — On node1, configure ansible and enable helm:")
print()
print("   cat > inventory/mycluster/group_vars/all/all.yml << 'EOF'")
print("ansible_user: cc")
print("ansible_ssh_private_key_file: /home/cc/.ssh/forkwise-key")
print("ansible_become: true")
print("ansible_become_method: sudo")
print("disable_ipv6_dns: true")
print("EOF")
print()
print("   sed -i 's/^helm_enabled: false/helm_enabled: true/' inventory/mycluster/group_vars/k8s_cluster/addons.yml")
print()

print("STEP I — On node1, test and run kubespray (~20 min):")
print()
print("   ansible -i inventory/mycluster/hosts.yaml all -m ping")
print("   tmux new -s kubespray")
print("   cd ~/kubespray && source .venv/bin/activate")
print("   ansible-playbook -i inventory/mycluster/hosts.yaml cluster.yml -b 2>&1 | tee /tmp/kubespray.log")
print("   # Detach: Ctrl+B then D   Reattach: tmux attach -t kubespray")
print()

print("STEP J — On node1, post-kubespray setup:")
print()
print("   mkdir -p ~/.kube && sudo cp /etc/kubernetes/admin.conf ~/.kube/config && sudo chown $(id -u):$(id -g) ~/.kube/config")
print("   curl https://raw.githubusercontent.com/helm/helm/main/scripts/get-helm-3 | bash")
print("   kubectl apply -f https://raw.githubusercontent.com/rancher/local-path-provisioner/master/deploy/local-path-storage.yaml")
print("   kubectl patch storageclass local-path -p '{\"metadata\":{\"annotations\":{\"storageclass.kubernetes.io/is-default-class\":\"true\"}}}'")
print("   kubectl apply -f https://github.com/kubernetes-sigs/metrics-server/releases/latest/download/components.yaml")
print("   kubectl -n kube-system patch deployment metrics-server --type='json' -p='[{\"op\":\"add\",\"path\":\"/spec/template/spec/containers/0/args/-\",\"value\":\"--kubelet-insecure-tls\"}]'")
print("   kubectl -n kube-system get configmap coredns -o yaml | sed 's|forward . /etc/resolv.conf|forward . 8.8.8.8 1.1.1.1|' | kubectl apply -f -")
print("   kubectl -n kube-system rollout restart deployment coredns")
print()

print("STEP K — Verify:")
print()
print("   kubectl get nodes")
print("   kubectl get pods -A")

---

## Teardown

Run these cells **only** when you want to destroy all infrastructure and free resources.

In [ ]:
# TEARDOWN: Destroy all Terraform-managed resources
# run_tf("terraform destroy -auto-approve", "Terraform Destroy")

In [ ]:
# TEARDOWN: Delete the lease
# l.delete()
# print("Lease deleted.")